In [1]:
from __future__ import annotations
import h5py as h5
import numpy as np

from pathlib import Path

from structure_tensor import structure_tensor_3d, eig_special_3d
from structure_tensor.st3d import eigh_baseline_3d, S6_to_mat33

from structure_tensor.pre_processing import normalize
## need something with padding
from structure_tensor.post_processing import align_direction

from structure_tensor.h5_io import H5BlockReader, H5BlockWriter

import tqdm

In [2]:
in_path = Path("../data2/original/285_01_HR_.vol.sub.h5")

out_path = Path("../results/285_01_HR_.vol.sub.vec.h5")

In [19]:
keys_in = ("volume",)  # add whatever you want
z_block = 128

# Inspect input shapes once to build output specs
with h5.File(in_path, "r") as F:
    obj = F["volume"]         # (Z,Y,X)
    
    if isinstance(obj,h5.Dataset):
        vol_shape = obj.shape
        # example output: eigenvector field (3,Z,Y,X) float32
        out_vec_shape = (3,) + vol_shape
        print(vol_shape)
    else:
        print("error")
specs = {
    "vec": dict(
        shape=out_vec_shape,
        dtype=np.float32,
        chunks=(3, z_block, 128, 128),      # tune to your access pattern
        compression="gzip",
        compression_opts=4,
    ),
    "vol": dict(
        shape=vol_shape,
        dtype=np.uint16,
        chunks=(z_block, 128, 128),      # tune to your access pattern
        compression="gzip",
        compression_opts=4,
    )
}

## Scan Parameters
voxel_size = 0.00230760 # voxel size in µm/Px
## Material Parameters
fiber_diameter = 7.e-3 # fiber diameter in µm

## set parameters for Gaussian Kernel
r = fiber_diameter / 2 / voxel_size
sigma = round(np.sqrt(r**2 / 2), 2)
rho = 4 * sigma

axes = ("y","z")

with H5BlockReader(in_path, keys_in, z_block=z_block, dtype=np.float32, strict=False) as reader, \
     H5BlockWriter(out_path, specs, mode="w") as writer:

    for zsl, batch in tqdm.tqdm(reader, desc="Blocks", unit="blk"):
        vol = batch.get("volume")  # (zb, Y, X)
        if vol is None:
            continue

        vol_norm = normalize(vol, method="robust")

        S = structure_tensor_3d(vol_norm, sigma, rho)
        S = S6_to_mat33(S)

        _, vec = eigh_baseline_3d(S, full=False)

        # Debug only if suspicious
        maxabs_pre = float(np.max(np.abs(vec)))
        if (not np.isfinite(vec).all()) or (maxabs_pre > 1.5):
            print(f"[WARN] pre-align: z={zsl} finite={np.isfinite(vec).all()} maxabs={maxabs_pre}")

        vec = align_direction(vec, axes=axes)

        # Safe renormalize + clamp
        l = np.linalg.norm(vec, axis=0, keepdims=True)
        vec = np.where(l > 1e-12, vec / l, 0.0)
        np.clip(vec, -1.0, 1.0, out=vec)

        maxabs_post = float(np.max(np.abs(vec)))
        if maxabs_post > 1.01:
            print(f"[WARN] post-norm: z={zsl} maxabs={maxabs_post}")

        writer.write_block("vec", zsl, vec.astype(np.float32, copy=False))
        writer.write_block("vol", zsl, vol.astype(np.float32, copy=False))

(512, 512, 512)


Blocks: 4blk [04:04, 61.13s/blk]


In [1]:
from pathlib import Path
from structure_tensor.xdmf_io import write_xdmf_for_h5

In [3]:
out_path = Path("../results/285_01_HR_.vol.sub.vec.h5")

voxel_size = 0.00230760 # voxel size in µm/Px
xmf = write_xdmf_for_h5(
    out_path,
    keys=["vol", "vec"],
    grid_key = "vol",
    spacing_xyz=(voxel_size,voxel_size,voxel_size),  # dx, dy, dz
    center="Node"
)
print("Open in ParaView:", xmf)

Open in ParaView: ../results/285_01_HR_.vol.sub.vec.xdmf
